# 7.05 Productivity Toolkits — Slack · Gmail · GitHub · Jira · Office 365

사무 생산성 도구 5종을 **toolkit 패턴**으로 에이전트에 주입한다. 각 toolkit 은 `.get_tools()` 로 여러 도구를 묶어 반환 — 한 번의 인스턴스화로 "이슈 조회 + 코멘트 작성 + PR 머지" 같은 관련 도구 세트를 한꺼번에 얻는다.

| Toolkit | 주요 도구 | 인증 |
|---------|----------|------|
| `SlackToolkit` | `send_message`, `get_channels`, `get_messages` | Bot Token (OAuth) |
| `GmailToolkit` | `send_gmail_message`, `search_gmail`, `get_gmail_thread` | OAuth (credentials.json) |
| `GitHubToolkit` | `get_issues`, `create_pr`, `get_pr`, `create_review_comment` | Personal Access Token + App |
| `JiraToolkit` | `jql_query`, `create_issue`, `comment` | API Token + Email |
| `O365Toolkit` | `send_email`, `search_events`, `create_event` | Azure AD App |

## 학습 목표

- Toolkit 인스턴스화 → `.get_tools()` → `create_agent(tools=...)` 3 단계 표준 패턴
- 각 벤더의 **환경변수** 와 OAuth 파일 위치
- 쓰기 액션(메시지 전송·이슈 생성) 을 **HITL** 로 감싸는 필수 패턴
- 실제 키가 없어도 toolkit 객체 자체는 생성 가능 — `.get_tools()` 로 노출되는 **도구 목록 탐색** 만 수행

## 언제 쓰나

- "어제 고객사 Slack 채널에서 언급된 이슈 요약해줘"
- "이 PR 의 리뷰 코멘트를 합쳐 요약하고 Jira 에 이슈로 옮겨줘"
- 사내 헬프데스크 봇이 Gmail/Slack/Jira 를 한 번에 오가며 티켓을 처리

## 7.05.1 환경 설정

5개 벤더 중 **하나라도 키가 있으면** 실습 가능. 없으면 도구 카탈로그만 확인하는 교육 모드.

> 드리프트 주의: `GitHubToolkit` 은 `langchain_community.agent_toolkits` top-level 에서 export 되지 않는다. 서브모듈 경로 필요.

In [ ]:
# !pip install -U langchain langchain-community slack_sdk google-api-python-client google-auth-httplib2 \
#     google-auth-oauthlib PyGithub atlassian-python-api O365

import os
from dotenv import load_dotenv
load_dotenv(override=True)

prod_keys = {
    "Slack":  bool(os.environ.get("SLACK_BOT_TOKEN") or os.environ.get("SLACK_USER_TOKEN")),
    "Gmail":  bool(os.environ.get("GMAIL_CREDS_JSON") or os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")),
    "GitHub": bool(os.environ.get("GITHUB_APP_ID") and os.environ.get("GITHUB_APP_PRIVATE_KEY") and os.environ.get("GITHUB_REPOSITORY")),
    "Jira":   bool(os.environ.get("JIRA_API_TOKEN") and os.environ.get("JIRA_USERNAME") and os.environ.get("JIRA_INSTANCE_URL")),
    "O365":   bool(os.environ.get("O365_CLIENT_ID") and os.environ.get("O365_CLIENT_SECRET")),
}
print(prod_keys)
# 교육용: 키가 하나도 없어도 아래 섹션은 toolkit import/카탈로그만 표시하는 모드로 진행
print("실습 가능 여부:", any(prod_keys.values()))

## 7.05.2 기본 사용 — 각 Toolkit 의 `get_tools()` 목록 탐색

키가 없어도 toolkit 객체는 **인스턴스화 가능**하다 (인증은 실제 호출 때 수행). 각 벤더가 내보내는 **도구 이름과 설명**을 먼저 탐색한다.

In [ ]:
# Slack
try:
    from langchain_community.agent_toolkits import SlackToolkit
    slack_kit = SlackToolkit()
    slack_tools = slack_kit.get_tools()
    print("[Slack] 도구:", [t.name for t in slack_tools])
except Exception as e:
    print("[Slack] 스킵 (패키지 또는 토큰 문제):", type(e).__name__, str(e)[:120])
    slack_tools = []

# Gmail — credentials.json 이 없으면 .get_tools() 는 되지만 일부 필드가 빈 상태
try:
    from langchain_community.agent_toolkits import GmailToolkit
    gmail_kit = GmailToolkit()
    gmail_tools = gmail_kit.get_tools()
    print("[Gmail] 도구:", [t.name for t in gmail_tools])
except Exception as e:
    print("[Gmail] 스킵:", type(e).__name__, str(e)[:120])
    gmail_tools = []

In [ ]:
# GitHub — top-level export 없음, 서브모듈에서 import
try:
    from langchain_community.agent_toolkits.github.toolkit import GitHubToolkit
    from langchain_community.utilities.github import GitHubAPIWrapper
    if prod_keys["GitHub"]:
        gh_wrapper = GitHubAPIWrapper()
        gh_kit = GitHubToolkit.from_github_api_wrapper(gh_wrapper)
        gh_tools = gh_kit.get_tools()
        print("[GitHub] 도구 수:", len(gh_tools), "— 처음 5개:", [t.name for t in gh_tools[:5]])
    else:
        print("[GitHub] 키 없음 — GitHubToolkit 은 GITHUB_APP_ID/PRIVATE_KEY/REPOSITORY 환경변수 필요")
except Exception as e:
    print("[GitHub] 스킵:", type(e).__name__, str(e)[:120])

# Jira
try:
    from langchain_community.agent_toolkits import JiraToolkit
    from langchain_community.utilities.jira import JiraAPIWrapper
    if prod_keys["Jira"]:
        jira_kit = JiraToolkit.from_jira_api_wrapper(JiraAPIWrapper())
        print("[Jira] 도구:", [t.name for t in jira_kit.get_tools()])
    else:
        print("[Jira] 키 없음 — JIRA_API_TOKEN/USERNAME/INSTANCE_URL 환경변수 필요")
except Exception as e:
    print("[Jira] 스킵:", type(e).__name__, str(e)[:120])

# O365
try:
    from langchain_community.agent_toolkits import O365Toolkit
    if prod_keys["O365"]:
        o365_kit = O365Toolkit()
        print("[O365] 도구:", [t.name for t in o365_kit.get_tools()])
    else:
        print("[O365] 키 없음 — O365_CLIENT_ID/SECRET + Azure AD App 등록 필요")
except Exception as e:
    print("[O365] 스킵:", type(e).__name__, str(e)[:120])

## 7.05.3 `create_agent` 에 연결 — Slack 읽기 전용 에이전트 (key-gated)

실제 키가 있을 때만 실행. 읽기 전용 도구(`get_channels`, `get_messages`)만 노출해 에이전트가 **메시지 조회**만 가능하게 한다. 쓰기(`send_message`)는 다음 섹션 HITL 로 감싼다.

In [ ]:
if prod_keys["Slack"] and os.environ.get("OPENAI_API_KEY") and slack_tools:
    from langchain.agents import create_agent

    read_only = [t for t in slack_tools if "send" not in t.name.lower()]
    print("읽기 전용 도구:", [t.name for t in read_only])

    agent = create_agent(
        model="openai:gpt-4.1-mini",
        tools=read_only,
        system_prompt="Slack 워크스페이스의 채널·메시지를 조회만 한다. 쓰기 작업은 절대 시도하지 마라.",
    )
    res = agent.invoke({"messages": [{"role": "user", "content": "사용 가능한 채널 목록을 보여줘."}]})
    print(res["messages"][-1].content[:400])
else:
    print("Slack + OpenAI 키 조합 없음 → 실행 스킵. 위 7.05.2 에서 도구 카탈로그만 확인.")

## 7.05.4 주요 옵션 · 비교 표

### 인증 · 환경변수

| Toolkit | 필수 환경변수 | 추가 설정 |
|---------|--------------|----------|
| Slack | `SLACK_BOT_TOKEN` (`xoxb-...`) | App scopes: `channels:read`, `chat:write`, `users:read` |
| Gmail | `GOOGLE_APPLICATION_CREDENTIALS` 또는 `get_gmail_credentials(token_file=...)` | OAuth 콜백 초기 승인 필요 |
| GitHub | `GITHUB_APP_ID`, `GITHUB_APP_PRIVATE_KEY`, `GITHUB_REPOSITORY`, (선택) `GITHUB_BRANCH` | GitHub App 설치 + repo 권한 |
| Jira | `JIRA_API_TOKEN`, `JIRA_USERNAME`(email), `JIRA_INSTANCE_URL` | Atlassian Cloud API Token |
| O365 | `O365_CLIENT_ID`, `O365_CLIENT_SECRET` | Azure AD App 등록, 콜백 URL |

### Scope · 권한 원칙

- **읽기 권한만** 먼저 승인 → 에이전트 동작 검증 → 필요시 쓰기 추가
- Slack 봇이 모든 채널 접근하지 않도록 **private channel 초대**만 허용
- GitHub App 권한은 repo 단위 — 조직 전역은 피할 것
- O365 app-only 권한은 광범위 — 가능하면 delegated 권한

## 7.05.5 (필수) HITL 결합 — 쓰기 액션 승인

생산성 toolkit 의 쓰기 도구는 **돌이킬 수 없다**. Slack 메시지·Jira 이슈·이메일은 전송 즉시 상대방에게 도달. 반드시 `HumanInTheLoopMiddleware` 로 감싸야 한다.

```python
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware

WRITE_TOOLS = {
    "send_message",         # Slack
    "send_gmail_message",   # Gmail
    "create_an_issue",      # GitHub
    "create_issue",         # Jira
    "send_event",           # O365
}

agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[*slack_tools, *gmail_tools],  # 실제 사용 조합으로
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            name: {"allowed_decisions": ["approve", "edit", "reject"]}
            for name in WRITE_TOOLS
        },
    )],
)
```

`edit` 으로 메시지 본문·수신자를 수정한 뒤 최종 발송. 전체 HITL resume 흐름과 Slack 미리보기 UI 연결 패턴은 `02_langchain/07_hitl_and_runtime` 노트북 참고.

## 체크리스트

- [ ] 5개 toolkit 중 **필요한 것만** 물린다 — 도구 20+ 개는 모델 라우팅 품질 급락
- [ ] 쓰기 도구(`send_*`, `create_*`, `delete_*`) 는 **예외 없이 HITL**
- [ ] 토큰 스코프는 최소 권한 — 읽기만 필요하면 쓰기 scope 금지
- [ ] OAuth 콜백 URL 은 로컬 개발 시 `http://localhost:*` 명시 필요 (Gmail, O365)
- [ ] `GitHubToolkit` 는 **App 방식만 지원** (PAT 단독 금지) — 운영 전환 비용 주의
- [ ] Slack bot 을 DM 봇처럼 쓰려면 `im:history`, `im:read`, `im:write` 스코프 추가

## 다음

- `08_integration/04_document_loaders/04_productivity.ipynb` — Slack export, GitHub repo, Confluence, Notion, Figma **파일 로더** 쪽
- `02_langchain/07_hitl_and_runtime` — `interrupt_on` + `Command(resume=...)` 전체 흐름

## 참고

- Slack Tool: https://python.langchain.com/docs/integrations/tools/slack/
- Gmail Tool: https://python.langchain.com/docs/integrations/tools/gmail/
- GitHub Tool: https://python.langchain.com/docs/integrations/tools/github/
- Jira Tool: https://python.langchain.com/docs/integrations/tools/jira/
- O365 Tool: https://python.langchain.com/docs/integrations/tools/office365/